# Telecom Customer Churn Analytics & Prediction

## Notebook 4: Machine Learning Model Building

### Objective

The objective of this notebook is to build multiple classification models to predict customer churn.

The models will be trained on the preprocessed dataset and compared using appropriate evaluation metrics to identify the best-performing model.

The models implemented are:

- Logistic Regression
- Decision Tree Classifier
- Random Forest Classifier
- K-Nearest Neighbors (KNN)
- Gaussian Naive Bayes

The best model will later be evaluated in detail using confusion matrix, ROC curve, precision, recall, F1-score, and feature importance.

## Import Libraries

In [12]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.model_selection import GridSearchCV
import joblib

In [2]:
from sklearn.metrics import accuracy_score

## Load Processed Data

In [4]:
train_df = pd.read_csv("../Data/processed/train_data.csv")

test_df = pd.read_csv("../Data/processed/test_data.csv")

In [5]:
X_train = train_df.drop("Churn", axis=1)

y_train = train_df["Churn"]

X_test = test_df.drop("Churn", axis=1)

y_test = test_df["Churn"]

In [6]:
print(X_train.shape)

print(X_test.shape)

print(y_train.shape)

print(y_test.shape)

(5634, 30)
(1409, 30)
(5634,)
(1409,)


## Creating Models

In [71]:
models={
    "Logistic Regression":LogisticRegression(
        max_iter=1000,
        random_state=42
    ),
    "Decision Tree":DecisionTreeClassifier(
        max_depth=5,
        min_samples_split=10,
        min_samples_leaf=5,
        random_state=42
    ),
    "Random Forest":RandomForestClassifier(
        n_estimators=200,
        max_depth=10,
        min_samples_split=10,
        min_samples_leaf=4,
        random_state=42
    ),
    "KNN":KNeighborsClassifier(
        n_neighbors=7,
        weights='distance'
    ),
    "Gaussian Naive Bayes": GaussianNB()
}

In [73]:
results=[]
predictions={}
trained_models={}

In [75]:
for name,model in models.items():
    # Train model
    model.fit(X_train,y_train)
    # Predict
    y_pred=model.predict(X_test)
    #Accuracy
    accuracy=accuracy_score(y_test,y_pred)
    # Store results
    results.append({
        'Model':name,
        'Accuracy':accuracy
    })
    # Store trained model
    trained_models[name]=model
    # Store predictions
    predictions[name]=y_pred
    print(f"{name} trained successfully!")

Logistic Regression trained successfully!
Decision Tree trained successfully!
Random Forest trained successfully!
KNN trained successfully!
Gaussian Naive Bayes trained successfully!


In [76]:
trained_models

{'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
 'Decision Tree': DecisionTreeClassifier(max_depth=5, min_samples_leaf=5, min_samples_split=10,
                        random_state=42),
 'Random Forest': RandomForestClassifier(max_depth=10, min_samples_leaf=4, min_samples_split=10,
                        n_estimators=200, random_state=42),
 'KNN': KNeighborsClassifier(n_neighbors=7, weights='distance'),
 'Gaussian Naive Bayes': GaussianNB()}

In [79]:
predictions

{'Logistic Regression': array([0, 1, 0, ..., 0, 0, 0], dtype=int64),
 'Decision Tree': array([0, 1, 0, ..., 0, 0, 0], dtype=int64),
 'Random Forest': array([0, 1, 0, ..., 0, 0, 0], dtype=int64),
 'KNN': array([0, 1, 0, ..., 0, 0, 0], dtype=int64),
 'Gaussian Naive Bayes': array([0, 1, 0, ..., 0, 0, 0], dtype=int64)}

In [81]:
results

[{'Model': 'Logistic Regression', 'Accuracy': 0.8062455642299503},
 {'Model': 'Decision Tree', 'Accuracy': 0.794180269694819},
 {'Model': 'Random Forest', 'Accuracy': 0.8048261178140526},
 {'Model': 'KNN', 'Accuracy': 0.758694109297374},
 {'Model': 'Gaussian Naive Bayes', 'Accuracy': 0.6557842441447835}]

In [83]:
results_df = pd.DataFrame(results)

results_df

,Model,Accuracy
0,Logistic Regression,0.806246
1,Decision Tree,0.794180
2,Random Forest,0.804826
3,KNN,0.758694
4,Gaussian Naive Bayes,0.655784


In [85]:
results_df=results_df.sort_values(by='Accuracy',ascending=False).reset_index(drop=True)

In [87]:
results_df

,Model,Accuracy
0,Logistic Regression,0.806246
1,Random Forest,0.804826
2,Decision Tree,0.794180
3,KNN,0.758694
4,Gaussian Naive Bayes,0.655784


### Best Model

In [90]:
best_model_name = results_df.loc[0, "Model"]

print(f"Best Model: {best_model_name}")

Best Model: Logistic Regression


In [92]:
results_df["Accuracy"] = results_df["Accuracy"].round(4)

results_df

,Model,Accuracy
0,Logistic Regression,0.8062
1,Random Forest,0.8048
2,Decision Tree,0.7942
3,KNN,0.7587
4,Gaussian Naive Bayes,0.6558


#### We'll tune both:

**✅ Logistic Regression**
**✅ Random Forest**

## **Hyperparameter Tuning**

### Hyperparameter Tuning - Logistic Regression

In [190]:
lr_param_grid = {
    'C': [0.01, 0.1, 1, 10, 100],
    'solver': ['liblinear', 'lbfgs']
}

In [192]:
lr_grid = GridSearchCV(
    estimator=LogisticRegression(
        random_state=42,
        max_iter=1000
    ),

    param_grid=lr_param_grid,

    scoring='f1',

    cv=5,

    n_jobs=-1,

    verbose=1
)

In [194]:
lr_grid.fit(X_train, y_train)

Fitting 5 folds for each of 10 candidates, totalling 50 fits


GridSearchCV(cv=5, estimator=LogisticRegression(max_iter=1000, random_state=42),
             n_jobs=-1,
             param_grid={'C': [0.01, 0.1, 1, 10, 100],
                         'solver': ['liblinear', 'lbfgs']},
             scoring='f1', verbose=1)

In [196]:
print("Best Parameters:")
print(lr_grid.best_params_)

Best Parameters:
{'C': 100, 'solver': 'liblinear'}


In [198]:
print("Best Cross Validation F1 Score:")
print(round(lr_grid.best_score_, 4))

Best Cross Validation F1 Score:
0.5999


In [200]:
best_lr = lr_grid.best_estimator_

In [202]:
lr_pred = best_lr.predict(X_test)

lr_accuracy = accuracy_score(y_test, lr_pred)

print(f"Tuned Logistic Regression Accuracy: {lr_accuracy:.4f}")

Tuned Logistic Regression Accuracy: 0.8020


### Conclusion on Logistic Regression model
- Hyperparameter tuning did not improve the Logistic Regression model's performance on the test dataset.
- Therefore, the baseline Logistic Regression model remains the preferred version.

- `Model	                Baseline Accuracy|	Tuned Accuracy|	    Keep`
- `Logistic Regression	    0.8062	         |      0.8020	  |  ✅ Baseline`

### Hyperparameter Tuning -Random Forest

In [204]:
rf_param_grid = {
    "n_estimators": [100, 200, 300],
    "max_depth": [10, 15, 20, None],
    "min_samples_split": [2, 5, 10],
    "min_samples_leaf": [1, 2, 4],
    "max_features": ["sqrt", "log2"]
}

In [206]:
rf_grid = GridSearchCV(

    estimator=RandomForestClassifier(
        random_state=42
    ),

    param_grid=rf_param_grid,

    scoring='f1',

    cv=5,

    n_jobs=-1,

    verbose=1

)

In [208]:
rf_grid.fit(X_train, y_train)

Fitting 5 folds for each of 216 candidates, totalling 1080 fits


GridSearchCV(cv=5, estimator=RandomForestClassifier(random_state=42), n_jobs=-1,
             param_grid={'max_depth': [10, 15, 20, None],
                         'max_features': ['sqrt', 'log2'],
                         'min_samples_leaf': [1, 2, 4],
                         'min_samples_split': [2, 5, 10],
                         'n_estimators': [100, 200, 300]},
             scoring='f1', verbose=1)

In [209]:
print("Best Parameters:")
print(rf_grid.best_params_)

Best Parameters:
{'max_depth': 10, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 300}


In [210]:
print("Best Cross Validation F1 Score:")
print(round(rf_grid.best_score_, 4))

Best Cross Validation F1 Score:
0.5707


In [211]:
best_rf = rf_grid.best_estimator_

In [212]:
rf_pred = best_rf.predict(X_test)

rf_accuracy = accuracy_score(y_test, rf_pred)

print(f"Tuned Random Forest Accuracy: {rf_accuracy:.4f}")

Tuned Random Forest Accuracy: 0.8034


### Compare Both Tuned Models

In [219]:
tuned_results = pd.DataFrame({

    'Model':[
        'Logistic Regression',
        'Random Forest'
    ],

    'CV F1 Score':[
        lr_grid.best_score_,
        rf_grid.best_score_
    ],

    'Test Accuracy':[
        lr_accuracy,
        rf_accuracy
    ]

})

tuned_results = tuned_results.sort_values(
    by='CV F1 Score',
    ascending=False
).reset_index(drop=True)

tuned_results

,Model,CV F1 Score,Test Accuracy
0,Logistic Regression,0.599884,0.801987
1,Random Forest,0.570683,0.803407


## Saving Models

In [222]:
import os

os.makedirs("../models",exist_ok=True)

joblib.dump(best_lr,"../models/logistic_regression.pkl")

joblib.dump(best_rf,"../models/random_forest.pkl")

print("Models saved successfully.")

Models saved successfully.


## Save Best Parameters

In [225]:
import json

best_parameters = {

    "Logistic Regression": lr_grid.best_params_,

    "Random Forest": rf_grid.best_params_

}

with open("../models/best_parameters.json","w") as file:

    json.dump(best_parameters,file,indent=4)

print("Best Parameters Saved Successfully.")

Best Parameters Saved Successfully.
